In [1]:
# ============================================================
# PACKAGE 8.4 : LEAKAGE-FREE FEATURE SELECTION PIPELINE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold

import joblib


print("="*75)
print("PACKAGE 8.4 : LEAKAGE-FREE FEATURE SELECTION PIPELINE")
print("="*75)



# ------------------------------------------------------------
# Load Model Ready Data
# ------------------------------------------------------------

X_train = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package3_Feature_Encoding\X_train_final.csv"
)

X_test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package3_Feature_Encoding\X_test_final.csv"
)


y_train = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package3_Feature_Encoding\y_train.csv"
).squeeze()


y_test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package3_Feature_Encoding\y_test.csv"
).squeeze()



print("\nOriginal Shapes")

print("X_train:", X_train.shape)

print("X_test :", X_test.shape)



# ------------------------------------------------------------
# Remove Duplicate Columns
# ------------------------------------------------------------

duplicate_columns = (
    X_train.columns[
        X_train.T.duplicated()
    ]
)


print("\nDuplicate Columns Removed:")

print(list(duplicate_columns))


X_train = X_train.drop(
    columns=duplicate_columns
)

X_test = X_test.drop(
    columns=duplicate_columns
)



# ------------------------------------------------------------
# Remove Constant Features
# ------------------------------------------------------------

variance_filter = VarianceThreshold(
    threshold=0
)


X_train_filtered = variance_filter.fit_transform(
    X_train
)


selected_variance_features = (
    X_train.columns[
        variance_filter.get_support()
    ]
)


X_train = pd.DataFrame(
    X_train_filtered,
    columns=selected_variance_features
)


X_test = X_test[
    selected_variance_features
]


print(
    "\nAfter Variance Filtering:"
)

print(
    X_train.shape
)



# ------------------------------------------------------------
# Correlation Analysis
# ------------------------------------------------------------

correlation = (
    pd.concat(
        [
            X_train,
            y_train
        ],
        axis=1
    )
    .corr()["AQI_Target"]
    .drop("AQI_Target")
    .sort_values(
        ascending=False
    )
)


correlation_df = pd.DataFrame(
    {
        "Feature":
            correlation.index,

        "Correlation":
            correlation.values
    }
)


print("\nTop Correlated Features")

print(
    correlation_df.head(20)
)



# ------------------------------------------------------------
# Random Forest Feature Importance
# TRAIN ONLY
# ------------------------------------------------------------

rf_selector = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)


rf_selector.fit(
    X_train,
    y_train
)



importance_df = pd.DataFrame(
    {
        "Feature":
            X_train.columns,

        "Importance":
            rf_selector.feature_importances_
    }
)


importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)



print("\nTop Feature Importance")

print(
    importance_df.head(20)
)



# ------------------------------------------------------------
# Select Top Features
# ------------------------------------------------------------

TOP_FEATURES = 30


selected_features = (
    importance_df
    .head(TOP_FEATURES)["Feature"]
    .tolist()
)



print("\nSelected Features Count:")
print(len(selected_features))


print("\nSelected Features:")
print(selected_features)



# ------------------------------------------------------------
# Apply Same Features to Test
# ------------------------------------------------------------

X_train_selected = X_train[
    selected_features
]


X_test_selected = X_test[
    selected_features
]



# ------------------------------------------------------------
# Satellite Importance Report
# ------------------------------------------------------------

satellite_features = [
    f for f in importance_df["Feature"]
    if (
        "MODIS" in f
        or
        "Sentinel" in f
    )
]


satellite_report = (
    importance_df[
        importance_df["Feature"]
        .isin(satellite_features)
    ]
)



print("\nSatellite Feature Importance")

print(
    satellite_report
)



# ------------------------------------------------------------
# Save Outputs
# ------------------------------------------------------------

X_train_selected.to_csv(
    "X_train_selected.csv",
    index=False
)


X_test_selected.to_csv(
    "X_test_selected.csv",
    index=False
)


importance_df.to_csv(
    "Feature_Importance_Report.csv",
    index=False
)


correlation_df.to_csv(
    "Feature_Correlation_Report.csv",
    index=False
)


satellite_report.to_csv(
    "Satellite_Feature_Importance_Report.csv",
    index=False
)


joblib.dump(
    selected_features,
    "Selected_Features.pkl"
)



print("\nSaved Files:")

print("1. X_train_selected.csv")
print("2. X_test_selected.csv")
print("3. Feature_Importance_Report.csv")
print("4. Feature_Correlation_Report.csv")
print("5. Satellite_Feature_Importance_Report.csv")
print("6. Selected_Features.pkl")


print("\nPackage 8.4 Completed Successfully.")

PACKAGE 8.4 : LEAKAGE-FREE FEATURE SELECTION PIPELINE

Original Shapes
X_train: (1826, 84)
X_test : (182, 84)

Duplicate Columns Removed:
[]

After Variance Filtering:
(1826, 83)

Top Correlated Features
          Feature  Correlation
0             AQI     0.898627
1         AQI_MA3     0.857890
2           PM2.5     0.849554
3            PM10     0.831432
4         AQI_MA7     0.822365
5        PM25_MA3     0.807460
6        AQI_Lag1     0.803617
7        PM10_MA3     0.782654
8        PM25_MA7     0.764306
9       PM25_Lag1     0.746127
10       PM10_MA7     0.737066
11       AQI_Lag3     0.718432
12      PM10_Lag1     0.708502
13       AQI_Lag7     0.656793
14        Benzene     0.641611
15            NO2     0.638686
16             NO     0.622697
17   Pressure_MA3     0.595281
18  Pressure_Lag1     0.590328
19       Pressure     0.578102

Top Feature Importance
                Feature  Importance
12                  AQI    0.468699
0                 PM2.5    0.346281
1            

In [3]:
selected_features = pd.read_pickle(r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package4_Feature_Selection\Selected_Features.pkl")
print(selected_features)

['AQI', 'PM2.5', 'PM10', 'Rainfall', 'Humidity_Change', 'PM10_Diff', 'AQI_MA3', 'NO2', 'Wind_Change', 'NOx', 'CO', 'NO', 'AQI_MA7', 'NH3', 'MODIS_AOD', 'PM25_Diff', 'O3', 'PM25_PM10_Ratio', 'Wind_Lag1', 'Wind_Humidity_Index', 'SO2', 'PM10_MA3', 'Temperature_Change', 'PM25_MA3', 'AQI_Lag7', 'AQI_Lag1', 'Benzene', 'Wind_Speed', 'CO_NO2_Ratio', 'MODIS_AOD_MA7']
